<a href="https://colab.research.google.com/github/RozieKii/localTest/blob/main/Universal_API_Components.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Ep5 Code Lab: Free APIs & Integration
# Google Colab version

!pip -q install requests

import requests
import json
from google.colab import userdata

# 1. CONFIG
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL_NAME = "openai/gpt-oss-20b"

# Read API key from Colab Secrets
# In Colab: click the key icon (Secrets) and create GROQ_API_KEY
API_KEY = userdata.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

print("Connected. Ready.")

# 2. QUICK TEST — raw request
def raw_call(prompt, model=MODEL_NAME):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "openai/gpt-oss-20b",
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(BASE_URL, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]

print(raw_call("Say hello in one short sentence."))

Connected. Ready.
Hello!


In [8]:

# Ep5: The 5 Universal API Components

# 1. BASE URL
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"

# 2. AUTH KEY
API_KEY = userdata.get("GROQ_API_KEY")

# 3. AUTH HEADER
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# 4. MODEL STRING + 5. MESSAGES LIST
payload = {
    "model": "openai/gpt-oss-20b",
    "messages": [
        {"role": "user", "content": "Hello"}
    ]
}

# Send request and inspect any error
response = requests.post(
    BASE_URL,
    headers=headers,
    json=payload,
    timeout=30
)

if response.ok:
    data = response.json()
    print(data["choices"][0]["message"]["content"])
else:
    print("Status:", response.status_code)
    print("Error details:", response.text)


Hello! How can I help you today?


In [9]:

# 4. THE MESSAGES LIST IN DEPTH

full_call = requests.post(
    BASE_URL,
    headers=headers,
    json={
        "model": "openai/gpt-oss-20b",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France?"}
        ]
    }
)

full_call.raise_for_status()
data = full_call.json()

print("Answer:", data["choices"][0]["message"]["content"])
print("Tokens used:", data.get("usage", {}).get("total_tokens", "Not returned"))


Answer: The capital of France is **Paris**.
Tokens used: 131


In [10]:

# 5. WRAPPER CLASS

class GroqClient:
    def __init__(self, api_key, base_url=BASE_URL,
                 default_model=MODEL_NAME):
        self.api_key = api_key
        self.base_url = base_url
        self.default_model = default_model

    def chat(self, messages, model=None, temperature=0):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": model or self.default_model,
            "messages": messages,
            "temperature": temperature
        }

        response = requests.post(
            self.base_url,
            headers=headers,
            json=payload,
            timeout=30
        )
        response.raise_for_status()
        data = response.json()

        return data["choices"][0]["message"]["content"]

client = GroqClient(api_key=API_KEY)
print("GroqClient ready!")


GroqClient ready!


In [11]:
# 6. TESTING THE WRAPPER

# Single call
answer = client.chat([
    {"role": "user", "content": "What is 2 + 2?"}
])
print("Single call:", answer)

# Multi-turn conversation
conversation = [
    {"role": "user", "content": "My name is Alex."},
    {"role": "assistant", "content": "Hello Alex!"},
    {"role": "user", "content": "What is my name?"}
]

print()
print("Multi-turn:", client.chat(conversation))

Single call: 2 + 2 = 4.

Multi-turn: Your name is Alex.


In [12]:

# 7. FIVE EPISODES INTO ONE PIPELINE

user_name = "Alex"

system_instruction = f"""
ROLE: You are a digital twin assistant for {user_name}.
KNOWLEDGE: Use available activity data from Ep1,
embeddings from Ep2, attention concepts from Ep3,
and prompting strategies from Ep4.
CONSTRAINTS: Recommend actions based only on data provided.
If data is missing, ask for more information.
TONE: Direct and analytical.
"""

twin_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next? I just finished studying."}
]

response = client.chat(twin_messages)
print(response)


I need more context to give a useful recommendation.  
Could you tell me:

1. What your short‑term goal is (e.g., relaxation, exercise, creative work).  
2. How much time you have available right now.  
3. Any pending tasks or commitments you’re aware of.  
4. Your current energy level or mood (e.g., high, moderate, low).  

With that information I can suggest the next best action.


In [13]:

# 8. STRETCH: TWO-TURN CONVERSATION

two_turn_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"},
    {"role": "assistant", "content": client.chat(twin_messages)},
    {"role": "user", "content": "Give me the top 3 priorities only."}
]

print(client.chat(two_turn_messages))


I don’t have enough information to determine your priorities.  
Could you share:

1. Your current goal or project.  
2. Any deadlines or time constraints.  
3. Your current energy or focus level.  

Once I have that, I’ll list the top three priorities.
